# SatQuery GeoChat-7B — Google Colab GPU Service

Upload this notebook to **Google Colab**, set runtime to **GPU** (T4 recommended), then **Runtime → Run all**.

## What this notebook does

1. Clones [SatQuery-AI](https://github.com/Sai-Vidyut/SatQuery-AI) and runs the **existing** `services/geochat` FastAPI service (real `MBZUAI/geochat-7B`, not the development mock).
2. Waits for model download/load on the Colab GPU.
3. Validates `/health` and runs a local VQA smoke test.
4. Opens an **ngrok** tunnel on port **8000** and prints the `GEOCHAT_*` variables for your Mac's `backend/.env`.

## Before you start

| Step | Action |
|------|--------|
| Runtime | **Runtime → Change runtime type → T4 GPU** + **High-RAM** strongly recommended (GeoChat load needs >12 GB system RAM) |
| HF token | Colab **Secrets** → add `HF_TOKEN` if the model requires Hugging Face auth |
| ngrok | Colab **Secrets** → add `NGROK_AUTHTOKEN` (from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)) |

**Port note:** Colab reserves **8080** for its Node process. GeoChat binds to **8000** only.

**Keep-alive:** The Colab session (and ngrok URL) must stay alive while your Mac uses GeoChat. SatQuery does **not** start GeoChat for you.


## CELL 1 — Title / environment check

Verifies Python, CUDA, and GPU memory. **Fails immediately** if no GPU is available.

**Success:** GPU name and VRAM printed, `CELL 1 PASSED`.


In [ ]:
# CELL 1 — Environment check
import platform
import subprocess

import torch

print("SatQuery GeoChat-7B — Google Colab GPU Service")
print("=" * 60)
print(f"python:         {platform.python_version()}")
print(f"torch:          {torch.__version__}")
print(f"cuda runtime:   {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available. In Colab: Runtime -> Change runtime type -> GPU, "
        "then restart the runtime and run all cells."
    )

idx = torch.cuda.current_device()
props = torch.cuda.get_device_properties(idx)
total_vram_gb = round(props.total_memory / (1024 ** 3), 2)
free_b, total_b = torch.cuda.mem_get_info(idx)
free_gb = round(free_b / (1024 ** 3), 2)

print(f"gpu index:      {idx}")
print(f"gpu name:       {props.name}")
print(f"gpu vram total: {total_vram_gb} GB")
print(f"gpu vram free:  {free_gb} GB")

if total_vram_gb < 12:
    print(f"WARNING: low VRAM ({total_vram_gb} GB). GeoChat 8-bit load may OOM on <15 GB.")

import psutil

vm = psutil.virtual_memory()
ram_total_gb = round(vm.total / (1024 ** 3), 2)
ram_avail_gb = round(vm.available / (1024 ** 3), 2)
print(f"system ram total: {ram_total_gb} GB")
print(f"system ram avail: {ram_avail_gb} GB")
if ram_total_gb < 20:
    print(
        "WARNING: standard Colab RAM (~12 GB) often OOM-kills GeoChat-7B during load (exit -9)."
    )
    print("Preferred: Runtime -> Change runtime type -> High-RAM, then restart and Run all.")
    print("Fallback: CELL 5 enables offload_folder + cpu=2GiB cap — slower but may work.")

if "T4" in props.name:
    print("GPU check: Tesla T4 detected (recommended).")
else:
    print(f"GPU check: using {props.name} (T4 not required if VRAM is sufficient).")

print("\n--- nvidia-smi ---")
subprocess.run(["nvidia-smi"], check=False)
print("\nCELL 1 PASSED: GPU environment OK.")


## CELL 2 — Get SatQuery repository

Clones or updates `https://github.com/Sai-Vidyut/SatQuery-AI` at `/content/SatQuery-AI`.

Defaults to branch **`sai/core-ai`** (override with env `SATQUERY_GIT_REF`).

Safe to rerun: pulls latest when the repo already exists.

**Success:** commit SHA printed, launcher and supervisor scripts found.


In [ ]:
# CELL 2 — Clone or update SatQuery-AI
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path("/content/SatQuery-AI")
REPO_URL = "https://github.com/Sai-Vidyut/SatQuery-AI.git"
GIT_REF = os.environ.get("SATQUERY_GIT_REF", "sai/core-ai")
LAUNCHER = REPO_ROOT / "services/geochat/scripts/colab_start.sh"
SUPERVISOR = REPO_ROOT / "services/geochat/scripts/colab_supervisor.py"
SERVICE_ROOT = REPO_ROOT / "services/geochat"
DIAGNOSTICS = REPO_ROOT / "services/geochat/scripts/colab_diagnostics.py"

if REPO_ROOT.is_dir() and (REPO_ROOT / ".git").is_dir():
    print(f"Repository present at {REPO_ROOT} — syncing {GIT_REF}...")
    subprocess.check_call(["git", "fetch", "origin", GIT_REF], cwd=str(REPO_ROOT))
    subprocess.check_call(["git", "checkout", GIT_REF], cwd=str(REPO_ROOT))
    subprocess.check_call(["git", "pull", "--ff-only", "origin", GIT_REF], cwd=str(REPO_ROOT))
else:
    print(f"Cloning {REPO_URL} (branch {GIT_REF}) -> {REPO_ROOT}")
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(REPO_ROOT)]
    )

os.chdir(REPO_ROOT)
required = {
    "launcher": LAUNCHER,
    "supervisor": SUPERVISOR,
    "service_root": SERVICE_ROOT / "geochat_service" / "main.py",
    "diagnostics": DIAGNOSTICS,
}
for label, path in required.items():
    if not path.exists():
        raise RuntimeError(f"Missing {label}: {path}")

LAUNCHER.chmod(0o755)
SUPERVISOR.chmod(0o755)

diag_src = DIAGNOSTICS.read_text(encoding="utf-8")
if "def service_startup_exited" not in diag_src:
    raise RuntimeError(
        f"{DIAGNOSTICS} is missing service_startup_exited on branch {GIT_REF}. "
        "Re-run CELL 2 after setting SATQUERY_GIT_REF=sai/core-ai (or merge latest main)."
    )

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT), text=True).strip()
short = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO_ROOT), text=True).strip()
branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=str(REPO_ROOT), text=True).strip()

print(f"cwd:          {Path.cwd()}")
print(f"git branch:   {branch}")
print(f"git ref:      {GIT_REF}")
print(f"git commit:   {commit}")
print(f"git short:    {short}")
print(f"launcher:     {LAUNCHER}")
print(f"supervisor:   {SUPERVISOR}")
print("\nCELL 2 PASSED: repository ready.")


## CELL 3 — Install / setup

Runs the repository's **`colab_start.sh --setup-only`** — installs pinned deps, clones GeoChat upstream, applies Phase 9B patches, verifies bitsandbytes.

Does **not** start uvicorn (that happens in CELL 5).

**If this fails:** check GPU runtime and read stderr output.


In [ ]:
# CELL 3 — Install / setup (repository launcher, --setup-only)
import gc
import os
import subprocess
from pathlib import Path

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Freed Python/GPU caches before setup.")

REPO_ROOT = Path("/content/SatQuery-AI")
LAUNCHER = REPO_ROOT / "services/geochat/scripts/colab_start.sh"
LOG_PATH = Path("/content/geochat_server.log")
SERVICE_PORT = 8000

os.chdir(REPO_ROOT)
env = os.environ.copy()
env.pop("GEOCHAT_SERVICE_FAKE_ENGINE", None)
env["GEOCHAT_MODEL_ID"] = "MBZUAI/geochat-7B"
env["GEOCHAT_EAGER_LOAD"] = "true"
env["GEOCHAT_SERVICE_HOST"] = "0.0.0.0"
env["GEOCHAT_PORT"] = str(SERVICE_PORT)
env["GEOCHAT_SERVICE_PORT"] = str(SERVICE_PORT)
env.setdefault("GEOCHAT_SRC", "/content/geochat")

# Propagate HF token if CELL 4 already ran in a prior session
if os.environ.get("HF_TOKEN"):
    env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    env.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

print("=== Running colab_start.sh --setup-only ===")
print("(deps, GeoChat clone, Phase 9B patches, bitsandbytes verify)")
setup_proc = subprocess.run(
    ["bash", str(LAUNCHER), "--setup-only"],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
if setup_proc.stdout:
    print(setup_proc.stdout)
if setup_proc.returncode != 0:
    if setup_proc.stderr:
        print(setup_proc.stderr)
    raise RuntimeError(f"Setup failed with exit code {setup_proc.returncode}")

LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("a") as log_fp:
    if setup_proc.stdout:
        log_fp.write(setup_proc.stdout)
    if setup_proc.stderr:
        log_fp.write(setup_proc.stderr)

print("\nCELL 3 PASSED: GeoChat dependencies and patches ready.")


## CELL 4 — Hugging Face auth

Reads `HF_TOKEN` from Colab Secrets (never printed).

If missing, continues with the public-download path when Hugging Face allows it.

**To add a secret:** sidebar key icon → Name `HF_TOKEN` → paste token → re-run this cell.


In [ ]:
# CELL 4 — Hugging Face authentication (optional)
import os

try:
    from google.colab import userdata

    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

HF_INSTRUCTIONS = """
Optional: add a Colab Secret for gated model downloads.
  1. Sidebar key icon (Secrets)
  2. Name: HF_TOKEN
  3. Value: your Hugging Face access token (never commit or share)
  4. Re-run this cell after adding the secret

If MBZUAI/geochat-7B downloads without auth on your account, you can skip HF_TOKEN.
"""

token = None
if IN_COLAB:
    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
else:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if token and str(token).strip():
    os.environ["HF_TOKEN"] = token.strip()
    os.environ["HUGGINGFACE_HUB_TOKEN"] = token.strip()
    print("HF_TOKEN: loaded from secrets/environment (value not printed)")
else:
    print(HF_INSTRUCTIONS.strip())
    print("\nHF_TOKEN: not set — continuing (public download path if allowed by Hugging Face).")

print("\nCELL 4 PASSED: Hugging Face auth step complete.")


## CELL 5 — Start GeoChat service

Starts **`colab_supervisor.py`** (background) which launches the existing `geochat_service` uvicorn app on **port 8000**.

- Reuses an already-healthy service if detected
- Stops stale supervisor/uvicorn before a fresh start
- Does not block forever

**Success:** supervisor PID and log path printed.


In [ ]:
# CELL 5 — Start GeoChat service (supervised background, non-blocking)
import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import httpx

REPO_ROOT = Path("/content/SatQuery-AI")
LAUNCHER = REPO_ROOT / "services/geochat/scripts/colab_start.sh"
SUPERVISOR = REPO_ROOT / "services/geochat/scripts/colab_supervisor.py"
LOG_PATH = Path("/content/geochat_server.log")
PID_PATH = Path("/content/geochat_server.pid")
EXIT_PATH = Path("/content/geochat_server.exit")
SUPERVISOR_PID_PATH = Path("/content/geochat_supervisor.pid")
STATE_PATH = Path("/content/geochat_colab_state.json")
SERVICE_PORT = 8000
SERVICE_URL = f"http://127.0.0.1:{SERVICE_PORT}"
MODEL_ID = "MBZUAI/geochat-7B"


def _is_alive(pid: int | None) -> bool:
    if pid is None:
        return False
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def _terminate_pid(pid: int, label: str, sig: int = signal.SIGTERM) -> None:
    try:
        os.kill(pid, sig)
        sig_name = "SIGTERM" if sig == signal.SIGTERM else "SIGKILL"
        print(f"Sent {sig_name} to {label} PID {pid}")
    except ProcessLookupError:
        print(f"{label} PID {pid} not running.")


def _wait_pid_gone(pid: int | None, label: str, timeout_s: int = 30) -> None:
    if pid is None:
        return
    for _ in range(timeout_s):
        if not _is_alive(pid):
            print(f"{label} PID {pid} stopped.")
            return
        time.sleep(1)
    if _is_alive(pid):
        print(f"{label} PID {pid} still alive after {timeout_s}s — sending SIGKILL.")
        _terminate_pid(pid, label, signal.SIGKILL)
        time.sleep(1)


def _stop_existing_supervised_service() -> None:
    """Stop old uvicorn/supervisor and wait before clearing artifacts.

    The previous supervisor writes /content/geochat_server.exit (often -15) when
    SIGTERM'd. If we start a new supervisor before the old one exits, that stale
    exit file can be mistaken for a new startup failure.
    """
    old_supervisor_pid = _read_supervisor_pid()
    old_service_pid = None
    if PID_PATH.exists():
        try:
            old_service_pid = int(PID_PATH.read_text().strip())
        except (OSError, ValueError):
            pass
    if old_service_pid is not None:
        _terminate_pid(old_service_pid, "service")
    if old_supervisor_pid is not None:
        _terminate_pid(old_supervisor_pid, "supervisor")
    _wait_pid_gone(old_service_pid, "service")
    _wait_pid_gone(old_supervisor_pid, "supervisor")
    for path in (PID_PATH, EXIT_PATH, SUPERVISOR_PID_PATH):
        path.unlink(missing_ok=True)
    time.sleep(0.5)


def _read_supervisor_pid() -> int | None:
    if not SUPERVISOR_PID_PATH.exists():
        return None
    try:
        return int(SUPERVISOR_PID_PATH.read_text().strip())
    except (OSError, ValueError):
        return None


def _try_health() -> dict | None:
    try:
        res = httpx.get(f"{SERVICE_URL}/health", timeout=5.0)
        if res.status_code == 200:
            return res.json()
    except httpx.HTTPError:
        return None
    return None


# Reuse a healthy already-running service
existing = _try_health()
if existing and existing.get("model_loaded") and existing.get("startup_state") == "ready":
    print("Existing GeoChat service is already READY — reusing without restart.")
    state = {
        "service_port": SERVICE_PORT,
        "service_url": SERVICE_URL,
        "model_id": MODEL_ID,
        "service_started": False,
        "service_reused": True,
    }
    STATE_PATH.write_text(json.dumps(state, indent=2))
    print(f"model_loaded={existing.get('model_loaded')} gpu={existing.get('gpu')}")
    print("\nCELL 5 PASSED: service already running.")
    raise SystemExit(0)

# Stop stale processes before a fresh start (must finish before new supervisor)
_stop_existing_supervised_service()

os.chdir(REPO_ROOT)
env = os.environ.copy()
env.pop("GEOCHAT_SERVICE_FAKE_ENGINE", None)
env["GEOCHAT_MODEL_ID"] = MODEL_ID
env["GEOCHAT_EAGER_LOAD"] = "true"
env["GEOCHAT_SERVICE_HOST"] = "0.0.0.0"
env["GEOCHAT_PORT"] = str(SERVICE_PORT)
env["GEOCHAT_SERVICE_PORT"] = str(SERVICE_PORT)
env.setdefault("GEOCHAT_SRC", "/content/geochat")
env["GEOCHAT_COLAB_MEMORY_PROFILE"] = "colab"
env["GEOCHAT_OFFLOAD_DIR"] = "/content/geochat_offload"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if os.environ.get("HF_TOKEN"):
    env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    env.setdefault("HUGGINGFACE_HUB_TOKEN", os.environ["HF_TOKEN"])

print("=== Starting colab_supervisor.py (uvicorn on port 8000) ===")
supervisor_launch_time = time.time()
supervisor_log = LOG_PATH.open("a")
supervisor_proc = subprocess.Popen(
    [
        sys.executable,
        str(SUPERVISOR),
        "--skip-setup",
        "--host",
        "0.0.0.0",
        "--port",
        str(SERVICE_PORT),
        "--log",
        str(LOG_PATH),
        "--pid-file",
        str(PID_PATH),
        "--exit-file",
        str(EXIT_PATH),
    ],
    cwd=str(REPO_ROOT),
    env=env,
    stdout=supervisor_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
SUPERVISOR_PID_PATH.write_text(str(supervisor_proc.pid))

print("Waiting for supervisor to write uvicorn PID (up to 60s)...")
for _ in range(60):
    if PID_PATH.exists():
        try:
            service_pid = int(PID_PATH.read_text().strip())
            if service_pid > 0:
                print(f"Service PID ready: {service_pid}")
                break
        except (OSError, ValueError):
            pass
    if EXIT_PATH.exists():
        try:
            exit_mtime = EXIT_PATH.stat().st_mtime
        except OSError:
            exit_mtime = supervisor_launch_time
        if exit_mtime < supervisor_launch_time - 0.5:
            print("Ignoring stale exit file from previous supervisor shutdown.")
            EXIT_PATH.unlink(missing_ok=True)
        else:
            exit_text = EXIT_PATH.read_text().strip()
            raise RuntimeError(
                f"GeoChat service exited during startup (exit={exit_text or 'unknown'}). "
                f"Inspect {LOG_PATH}"
            )
    time.sleep(1)
else:
    print("WARNING: service PID not written yet — CELL 6 will keep polling.")

state = {
    "service_port": SERVICE_PORT,
    "service_url": SERVICE_URL,
    "model_id": MODEL_ID,
    "service_started": True,
    "service_reused": False,
    "supervisor_pid": supervisor_proc.pid,
    "supervisor_launch_time": supervisor_launch_time,
}
STATE_PATH.write_text(json.dumps(state, indent=2))

print("=== GeoChat service supervisor launched ===")
print(f"Supervisor PID: {supervisor_proc.pid} -> {SUPERVISOR_PID_PATH}")
print(f"Logs:           {LOG_PATH}")
print(f"Listen:         0.0.0.0:{SERVICE_PORT}")
print(f"Endpoints:      GET /health  POST /v1/vqa  POST /v1/caption")
print("\nCELL 5 PASSED: supervisor running (model loads in CELL 6).")


## CELL 6 — Startup monitor

Polls `http://127.0.0.1:8000/health` for up to **20 minutes** using `service_startup_exited()` (90s grace — no false abort while PID file is pending).

Shows startup_state, GPU memory, process RSS, elapsed time, and log tail. Polls every 20s while `startup_state=starting`.

**If exit code -9:** Colab ran out of system RAM — use **High-RAM** runtime and re-run from CELL 1.

**Expected:** `starting` → `ready` (or `failed` with diagnostics).


In [ ]:
# CELL 6 — Startup monitor (poll /health up to 20 minutes)
import json
import os
import sys
import time
from pathlib import Path

import httpx

REPO_ROOT = Path("/content/SatQuery-AI")
SCRIPTS_DIR = REPO_ROOT / "services/geochat/scripts"
SERVICE_PORT = 8000
SERVICE_URL = f"http://127.0.0.1:{SERVICE_PORT}"
LOG_PATH = Path("/content/geochat_server.log")
PID_PATH = Path("/content/geochat_server.pid")
EXIT_PATH = Path("/content/geochat_server.exit")
SUPERVISOR_PID_PATH = Path("/content/geochat_supervisor.pid")
STATE_PATH = Path("/content/geochat_colab_state.json")
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from colab_diagnostics import (  # noqa: E402
    get_gpu_memory_mb,
    get_process_state,
    is_pid_alive,
    log_tail,
    print_failure_report,
    read_exit_code,
)
try:
    from colab_diagnostics import service_startup_exited  # noqa: E402
except ImportError:
    print("WARNING: service_startup_exited missing from repo — using notebook fallback.")
    print("Re-run CELL 2 to sync branch sai/core-ai (SATQUERY_GIT_REF).")

    def service_startup_exited(
        *,
        pid_path=PID_PATH,
        exit_path=EXIT_PATH,
        supervisor_pid=None,
        started_at=None,
        grace_s=90.0,
    ):
        exit_code = read_exit_code(exit_path)
        if exit_code is not None:
            return True
        service = get_process_state(pid_path)
        supervisor_alive = is_pid_alive(supervisor_pid) if supervisor_pid is not None else False
        if service.pid is not None and not service.alive:
            return True
        if supervisor_alive or service.alive:
            return False
        if started_at is None:
            return False
        return (time.time() - started_at) >= grace_s

TIMEOUT_S = 20 * 60
POLL_S = 5
POLL_S_LOADING = 20
STARTUP_GRACE_S = 90.0
LOG_TAIL_LINES = 15
MODEL_ID = "MBZUAI/geochat-7B"


def _supervisor_pid() -> int | None:
    if not SUPERVISOR_PID_PATH.exists():
        return None
    try:
        return int(SUPERVISOR_PID_PATH.read_text().strip())
    except (OSError, ValueError):
        return None


def _supervisor_launch_time() -> float | None:
    if not STATE_PATH.exists():
        return None
    try:
        value = json.loads(STATE_PATH.read_text()).get("supervisor_launch_time")
        return float(value) if value is not None else None
    except (OSError, ValueError, TypeError):
        return None


def _clear_stale_exit_file() -> None:
    launch_t = _supervisor_launch_time()
    if launch_t is None or not EXIT_PATH.exists():
        return
    try:
        if EXIT_PATH.stat().st_mtime < launch_t - 0.5:
            print("Clearing stale exit file from prior supervisor shutdown.")
            EXIT_PATH.unlink(missing_ok=True)
    except OSError:
        pass


def _process_exited(started_at: float) -> bool:
    _clear_stale_exit_file()
    return service_startup_exited(
        pid_path=PID_PATH,
        exit_path=EXIT_PATH,
        supervisor_pid=_supervisor_pid(),
        started_at=started_at,
        grace_s=STARTUP_GRACE_S,
    )


def _abort(reason: str) -> None:
    exit_code = read_exit_code(EXIT_PATH)
    print(f"\nERROR: {reason}")
    print(f"exit_code: {exit_code if exit_code is not None else '(not recorded)'}")
    if exit_code == -9:
        print(
            "\nLikely cause: Linux OOM killer (SIGKILL) during HF download / 8-bit model load."
        )
        print("Colab standard runtime (~12 GB system RAM) is often insufficient for GeoChat-7B.")
        print("Fix 1 (best): Runtime -> Change runtime type -> High-RAM, restart, Run all from CELL 1.")
        print("Fix 2: Re-run CELL 2 to pull latest sai/core-ai (offload_folder + cpu=2GiB cap).")
        print("       Log should show: max_memory caps: {0: '12GiB', 'cpu': '2GiB'} and offload_folder")
        print("Alternatives: Kaggle (30 GB RAM), local GPU, or GEOCHAT_VQA_PROVIDER=development on Mac.")
    if EXIT_PATH.exists():
        print(f"exit file: {EXIT_PATH.read_text().strip()}")
    print(f"\n--- last 50 log lines ({LOG_PATH}) ---")
    print(log_tail(LOG_PATH, 50))
    print_failure_report(title="GeoChat startup failure (CELL 6)")
    raise RuntimeError(reason)


# Fast path: already ready
try:
    res = httpx.get(f"{SERVICE_URL}/health", timeout=10.0)
    body = res.json()
    if res.status_code == 200 and body.get("model_loaded") and body.get("startup_state") == "ready":
        print("Service already READY — skipping wait loop.")
        state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
        state.update({"health": body, "startup_monitor_passed": True})
        STATE_PATH.write_text(json.dumps(state, indent=2))
        print("\nCELL 6 PASSED: model loaded and healthy.")
        raise SystemExit(0)
except httpx.HTTPError:
    pass

if not SUPERVISOR_PID_PATH.exists():
    raise RuntimeError("Supervisor PID file missing. Run CELL 5 first.")

_clear_stale_exit_file()

print(
    f"Waiting for {SERVICE_URL}/health "
    f"(timeout {TIMEOUT_S // 60} min; first download can take 10+ min)..."
)
started_at = time.time()
deadline = time.time() + TIMEOUT_S
last_body = None
poll_n = 0

while time.time() < deadline:
    elapsed_min = int((time.time() - started_at) // 60)
    elapsed_sec = int((time.time() - started_at) % 60)

    if _process_exited(started_at):
        _abort("GeoChat service process exited during startup.")

    service = get_process_state(PID_PATH)
    sup_pid = _supervisor_pid()
    sup_alive = is_pid_alive(sup_pid) if sup_pid else False
    gpu = get_gpu_memory_mb()
    print(
        f"  [{elapsed_min:02d}:{elapsed_sec:02d}] service_pid={service.pid} alive={service.alive} "
        f"rss_mb={service.rss_mb} supervisor_alive={sup_alive}"
    )
    if gpu.get("available"):
        print(
            f"           gpu_alloc_mb={gpu.get('allocated_mb')} "
            f"gpu_reserved_mb={gpu.get('reserved_mb')} total_mb={gpu.get('total_mb')}"
        )

    try:
        res = httpx.get(f"{SERVICE_URL}/health", timeout=10.0)
        last_body = res.json()
        print(
            f"           health status={last_body.get('status')} "
            f"startup_state={last_body.get('startup_state')} "
            f"model_loaded={last_body.get('model_loaded')} gpu={last_body.get('gpu')}"
        )
        if last_body.get("load_error"):
            print(f"           load_error={last_body.get('load_error')}")
        if last_body.get("startup_state") == "failed":
            _abort(f"GeoChat model load failed: {last_body.get('load_error') or 'see logs'}")
        if res.status_code == 200 and last_body.get("model_loaded") and last_body.get("startup_state") == "ready":
            state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
            state.update({"health": last_body, "startup_monitor_passed": True})
            STATE_PATH.write_text(json.dumps(state, indent=2))
            print("\nCELL 6 PASSED: model loaded and healthy.")
            break
    except httpx.HTTPError as exc:
        print(f"           health not reachable yet ({exc})")

    poll_n += 1
    if poll_n % 2 == 0:
        tail = log_tail(LOG_PATH, LOG_TAIL_LINES)
        if tail and tail != "(log file missing)":
            print(f"  [log tail]\n{tail}")

    if _process_exited(started_at):
        _abort("GeoChat service process exited during startup.")

    poll_s = POLL_S_LOADING if (last_body or {}).get("startup_state") == "starting" else POLL_S
    time.sleep(poll_s)
else:
    if last_body:
        print("Last /health body:", json.dumps(last_body, indent=2))
    _abort(f"Timed out after {TIMEOUT_S // 60} minutes waiting for model_loaded=true.")


## CELL 7 — Health check

Pretty-prints `/health` and verifies `model_name == MBZUAI/geochat-7B`.


In [ ]:
# CELL 7 — Health check
import json
from pathlib import Path

import httpx

SERVICE_URL = "http://127.0.0.1:8000"
STATE_PATH = Path("/content/geochat_colab_state.json")
MODEL_ID = "MBZUAI/geochat-7B"

res = httpx.get(f"{SERVICE_URL}/health", timeout=30.0)
res.raise_for_status()
body = res.json()

print(json.dumps(body, indent=2))

assert body.get("status") == "ok", f"Unhealthy status: {body.get('status')}"
assert body.get("model_loaded") is True, "model_loaded is not true"
assert body.get("startup_state") == "ready", f"startup_state not ready: {body.get('startup_state')}"
assert body.get("model_name") == MODEL_ID, f"Unexpected model_name: {body.get('model_name')}"
assert body.get("provider") == "geochat_service"

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
state["health"] = body
state["local_health_passed"] = True
STATE_PATH.write_text(json.dumps(state, indent=2))

print("\nCELL 7 PASSED: /health OK, model_name verified.")


## CELL 8 — Local VQA smoke test

Creates a **synthetic** PNG in Colab (no user upload). Calls `POST /v1/vqa`.

**Service smoke test only** — does not prove model quality on real satellite imagery.


In [ ]:
# CELL 8 — Local VQA smoke test (synthetic image)
import base64
import io
import json
from pathlib import Path

import httpx
from PIL import Image

SERVICE_URL = "http://127.0.0.1:8000"
STATE_PATH = Path("/content/geochat_colab_state.json")
MODEL_ID = "MBZUAI/geochat-7B"
QUESTION = "What land-cover types are visible in this image?"

print("Service smoke test only — synthetic RGB patch, not a real satellite scene.")

img = Image.new("RGB", (256, 256), color=(34, 139, 34))
# simple land/water pattern
for x in range(256):
    for y in range(128, 256):
        img.putpixel((x, y), (30, 100, 180))
buf = io.BytesIO()
img.save(buf, format="PNG")
raw = buf.getvalue()
width, height = img.size

payload = {
    "model_id": MODEL_ID,
    "question": QUESTION,
    "image": {
        "content_base64": base64.b64encode(raw).decode("ascii"),
        "format": "png",
        "filename": "smoke_synthetic.png",
    },
    "image_metadata": {
        "image_id": "colab-smoke-synthetic",
        "modality": "optical",
        "width": width,
        "height": height,
        "georeferenced": False,
        "benchmark_dataset": True,
    },
    "parameters": {"max_new_tokens": 128, "temperature": 1.0, "do_sample": False},
}

res = httpx.post(f"{SERVICE_URL}/v1/vqa", json=payload, timeout=600.0)
print(f"HTTP status: {res.status_code}")
if res.status_code != 200:
    raise RuntimeError(res.text[:1000])

body = res.json()
answer = (body.get("answer") or "").strip()
print(f"model_name:            {body.get('model_name')}")
print(f"runtime_ms:            {body.get('runtime_ms')}")
print(f"confidence_available:  {body.get('confidence_available')}")
print(f"confidence:            {body.get('confidence')}")
print("\nANSWER:")
print(answer)

mock_markers = ("development mock", "[fake geochat service]")
if any(m in answer.lower() for m in mock_markers):
    raise RuntimeError("Response looks like a mock/fake engine.")

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
state["vqa_smoke_passed"] = True
state["vqa_smoke_answer_preview"] = answer[:200]
STATE_PATH.write_text(json.dumps(state, indent=2))

print("\nCELL 8 PASSED: VQA smoke test complete (service smoke test only).")


## CELL 9 — ngrok setup

Installs `pyngrok`, reads **`NGROK_AUTHTOKEN`** from Colab Secrets (never printed), tunnels **port 8000**.

Prints `GEOCHAT_SERVICE_URL=https://...`


In [ ]:
# CELL 9 — ngrok tunnel to port 8000
import json
import os
from pathlib import Path

SERVICE_PORT = 8000
STATE_PATH = Path("/content/geochat_colab_state.json")

try:
    from google.colab import userdata

    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

NGROK_HELP = """
ngrok requires an auth token for tunnels.

Add a Colab Secret:
  1. Sidebar key icon (Secrets)
  2. Name: NGROK_AUTHTOKEN
  3. Value: your token from https://dashboard.ngrok.com/get-started/your-authtoken
  4. Re-run this cell

Token is never printed by this notebook.
"""

ngrok_token = None
if IN_COLAB:
    for key in ("NGROK_AUTHTOKEN", "NGROK_TOKEN"):
        try:
            ngrok_token = userdata.get(key)
            if ngrok_token:
                break
        except Exception:
            continue
else:
    ngrok_token = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_TOKEN")

if not ngrok_token or not str(ngrok_token).strip():
    print(NGROK_HELP.strip())
    raise RuntimeError("NGROK_AUTHTOKEN Colab Secret is required for public tunnel.")

get_ipython().system("pip install -q pyngrok")
from pyngrok import conf, ngrok

conf.get_default().auth_token = ngrok_token.strip()
conf.get_default().region = "us"

try:
    tunnel = ngrok.connect(SERVICE_PORT, bind_tls=True)
except Exception as exc:
    raise RuntimeError(f"ngrok tunnel failed: {exc}") from exc

public_url = getattr(tunnel, "public_url", str(tunnel)).rstrip("/")
if not public_url.startswith("http"):
    public_url = f"https://{public_url}"

print(f"GEOCHAT_SERVICE_URL={public_url}")

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
state["public_url"] = public_url
state["ngrok_connected"] = True
STATE_PATH.write_text(json.dumps(state, indent=2))

print("\nCELL 9 PASSED: ngrok tunnel active on port 8000.")


## CELL 10 — Public health test

Calls `GET {public_url}/health` through the ngrok tunnel.


In [ ]:
# CELL 10 — Public health test via ngrok
import json
from pathlib import Path

import httpx

STATE_PATH = Path("/content/geochat_colab_state.json")
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
public_url = state.get("public_url")
if not public_url:
    raise RuntimeError("public_url missing — run CELL 9 first.")

health_url = f"{public_url.rstrip('/')}/health"
print(f"GET {health_url}")

res = httpx.get(health_url, timeout=30.0)
res.raise_for_status()
body = res.json()

print(json.dumps(body, indent=2))
ready = (
    body.get("status") == "ok"
    and body.get("model_loaded") is True
    and body.get("startup_state") == "ready"
)

state["public_health"] = body
state["public_health_passed"] = ready
STATE_PATH.write_text(json.dumps(state, indent=2))

print(f"\nPublic URL:     {public_url}")
print(f"Health status:  {body.get('status')}")
print(f"Model loaded:   {body.get('model_loaded')}")
print(f"Service ready:  {ready}")
print("\nCELL 10 PASSED: public /health reachable.")


## CELL 11 — SatQuery backend config

Copy-paste block for your Mac `backend/.env`.


In [ ]:
# CELL 11 — SatQuery backend configuration (copy to Mac backend/.env)
import json
from pathlib import Path

STATE_PATH = Path("/content/geochat_colab_state.json")
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
public_url = state.get("public_url")
if not public_url:
    raise RuntimeError("public_url missing — run CELL 9 first.")

MODEL_ID = "MBZUAI/geochat-7B"
block = f"""GEOCHAT_VQA_PROVIDER=geochat_service
GEOCHAT_SERVICE_URL={public_url.rstrip('/')}
GEOCHAT_MODEL_ID={MODEL_ID}
GEOCHAT_SERVICE_TIMEOUT_S=120"""

print("Copy the following into your Mac SatQuery backend/.env file:\n")
print("-" * 60)
print(block)
print("-" * 60)
print("\nNotes:")
print("- Do not commit .env or share the ngrok URL publicly.")
print("- Restart your local backend after updating .env.")
print("- SatQuery uses GEOCHAT_VQA_PROVIDER=geochat_service for real GPU inference.")
print("\nCELL 11 PASSED: backend config block printed.")


## CELL 12 — Optional real backend validation (Mac)

Prints the exact pytest command for `backend/tests/test_phase14_geochat_service.py`.


In [ ]:
# CELL 12 — Optional real backend validation (run on your Mac)
import json
from pathlib import Path

STATE_PATH = Path("/content/geochat_colab_state.json")
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
public_url = state.get("public_url", "<your-ngrok-url>")

cmd = f"""cd backend && GEOCHAT_REAL_SERVICE_TEST=true GEOCHAT_SERVICE_URL={public_url.rstrip('/')} uv run pytest tests/test_phase14_geochat_service.py -k real_service -v"""

print("Colab cannot run your Mac backend. After updating backend/.env, run on your Mac:\n")
print(cmd)
print("\nThis uses the gated integration test in:")
print("  backend/tests/test_phase14_geochat_service.py")
print("\nCELL 12: optional Mac validation command printed.")


## CELL 13 — Keep-alive / operations

Session and tunnel guidance.


In [ ]:
# CELL 13 — Keep-alive / operations
print("""
SatQuery GeoChat on Colab — operations guide

1. Keep this Colab tab/session alive while your Mac uses GeoChat.
2. The ngrok public URL can change when you restart the tunnel or Colab runtime.
3. If Colab disconnects, re-run CELL 5 onward (or Run all) to restart the service and tunnel.
4. SatQuery backend does NOT start GeoChat — you must run this notebook (or equivalent) separately.
5. Local service listens on http://127.0.0.1:8000 inside Colab only; your Mac reaches it via ngrok.

Rerun tips:
- CELL 2: safe to pull latest repo
- CELL 3: safe to rerun setup (idempotent)
- CELL 5: detects an already-healthy service and reuses it
- CELL 9: creates a new tunnel URL — update backend/.env after each new tunnel

Logs: /content/geochat_server.log
Exit:  /content/geochat_server.exit (present if uvicorn crashed)
""")


## CELL 14 — Final status

Summary and next steps on your Mac.


In [ ]:
# CELL 14 — Final status
import json
from pathlib import Path

import httpx

STATE_PATH = Path("/content/geochat_colab_state.json")
LOG_PATH = Path("/content/geochat_server.log")
EXIT_PATH = Path("/content/geochat_server.exit")
MODEL_ID = "MBZUAI/geochat-7B"
SERVICE_URL = "http://127.0.0.1:8000"

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}

service_ready = False
local_health = "FAIL"
gpu_name = "(unknown)"
health_body = {}

try:
    res = httpx.get(f"{SERVICE_URL}/health", timeout=10.0)
    health_body = res.json()
    service_ready = (
        res.status_code == 200
        and health_body.get("model_loaded")
        and health_body.get("startup_state") == "ready"
    )
    local_health = "PASS" if service_ready else "FAIL"
    gpu_name = health_body.get("gpu") or gpu_name
except Exception as exc:
    local_health = f"FAIL ({exc})"

public_url = state.get("public_url", "(not configured — run CELL 9)")
public_ok = state.get("public_health_passed", False)

print("=" * 60)
print("SatQuery GeoChat — final status")
print("=" * 60)
print(f"GeoChat service:     {'READY' if service_ready else 'FAILED'}")
print(f"Model:               {MODEL_ID}")
print(f"GPU:                 {gpu_name}")
print(f"Local health:        {local_health}")
print(f"Public tunnel:       {public_url}")
print(f"Public health:       {'PASS' if public_ok else 'FAIL / not run'}")
print(f"VQA smoke:           {'PASS' if state.get('vqa_smoke_passed') else 'not run'}")
print(f"Server log:          {LOG_PATH}")
if EXIT_PATH.exists():
    print(f"Server exit code:    {EXIT_PATH.read_text().strip()}")

print("\nSatQuery backend configuration:")
if public_url.startswith("http"):
    print(f"  GEOCHAT_VQA_PROVIDER=geochat_service")
    print(f"  GEOCHAT_SERVICE_URL={public_url.rstrip('/')}")
    print(f"  GEOCHAT_MODEL_ID={MODEL_ID}")
    print(f"  GEOCHAT_SERVICE_TIMEOUT_S=120")
else:
    print("  Run CELL 9–11 to generate the tunnel URL and .env block.")

print("\nNext on your Mac:")
print("  1. Paste the GEOCHAT_* block into backend/.env")
print("  2. Restart the SatQuery backend")
print("  3. Run the optional pytest command from CELL 12")
print("  4. Keep this Colab session alive while using SatQuery")
print("=" * 60)
